#  Supercomputing Internet LLM Fine-Tuning LoRA Example

This notebook is an English, GitHub-friendly translation of the original document. The Python code cells are kept unchanged as requested.


## Overview

This article explains how to use  nodes to fine-tune the model `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`.

The goal is to introduce , cloud deployment, and fast development code for small and medium-sized enterprises.

This is public-interest code, generated with the help of DeepSeek and tested in an example environment.


## 1. Market context for large language models

Outside of API usage from major frontier vendors, most civilian LLM research and applications are already clearly differentiated at the 500B scale and below (excluding TAALAS-related techniques).

1. **Hundreds-of-billions-scale models (100B+ to 500B)** represented by OpenAI GPT-OSS-120B. This track currently emphasizes low bit precision, such as 4-bit native training precision, strong information encoding, and improved information efficiency without sacrificing quality as the baseline. For example, GPT-OSS-120B can be used as a benchmark and compared directly with GPT-4-class models. Without strong model optimization or theoretical support, domestic models in China may not challenge trillion-parameter models; even if they do, they may still find it difficult to compete with GPT-OSS-120B. This reflects the current mathematical and practical limitations of large-model development in China, not a denial of 10T- or 100T-parameter models. However, hundreds-of-billions-scale models usually require multi-GPU operation, which creates a technical barrier.

2. **Tens-of-billions-scale models** represented by OpenAI GPT-OSS-20B. This group includes the classic Microsoft Phi-4 series at 14B and 7B, as well as distilled models from the Qwen and DeepSeek families, which are mainstream offerings from first- and second-tier vendors. With industrial 4-bit support, these models can run directly on consumer-grade computers (30–50 GB). Their token generation speed is comparable to a single user's information-processing speed. In particular, their fine-tuning compute requirements are relatively small, making them suitable for small and medium-sized enterprises to perform secondary development directly at the model layer. In some cases, the cost can be as low as a few thousand RMB. After efficiency improvements or sub-4-bit optimization, they may become mainstream for edge computing.

3. **Billion-scale models** represented by DeepSeek. A classic example is the DeepSeek 1.5B distilled model, which usually occupies 3 GB of memory or less. After further optimization, these models can be deployed directly on mobile phones or small computers. Their technical parameters are comparable to tens-of-billions-scale models, making them suitable for teaching purposes with extremely low training cost, typically within tens of RMB. The techniques learned here can be quickly transferred to tens-of-billions-scale model workflows, making this an effective way to experiment and iterate.


## 2.  setup steps

The following steps show how to use an  notebook environment for fine-tuning.

1. Register for :  
   `https://www....`  
   `https://www....`

2. Click the red button in the upper-right corner labeled **Console**.

3. Click **Service Navigation** → **Artificial Intelligence** (blue button).  
   This opens the interface needed for the AI Notebook:  
   `https://www..../ui/console/index.html#/notebook`  
   `https://www..../ui/console/index.html#/notebook`

4. Click **Billing** in the upper-right corner → **Overview**.

5. Click **Top Up** → **Alipay**.

   Choose the top-up amount according to your service needs. The following example consumes less than 10 RMB (about 2 RMB in actual testing).

6. After topping up, return to the AI Notebook page:  
   `https://www..../ui/console/index.html#/notebook`

7. Click **Notebook** → **Create Notebook**, choose ................................................., and set **1 accelerator card**. Use the 4090 accelerator card with 24 GB memory, which helps avoid debugging issues for beginners.

8. Click **Development Image** → **Base Image** → **Framework Name: PyTorch** → **Framework Version: 2.6.0** → **Python Version: py3.12-ubuntu22.04** → **CUDA/DTK Version: cuda12.4**. Then click the red **Create** button in the lower-right corner.

   This automatically creates the environment and switches back to the Notebook interface, where you can work directly in Jupyter Notebook or log in through VS Code via Remote SSH.

9. Click **Quick Tools** → **JupyterLab**.

   This loads the interface. Usually, the system allows network access automatically. If it does not, contact support.

10. Click **root** → **Notebook** → **Python3**.

    It is also recommended to click the `+` tab at the top center and open **Other** → **Terminal**.

    Note: the container uses the `root` account by default.

11. In the terminal, enter the following (the code can be copied and pasted directly):

   `pip install --upgrade pip`

   If that does not work, switch to the Aliyun mirror:

   `pip config set global.index-url https://mirrors.aliyun.com/pypi/simple/`  
   `pip install --upgrade pip`

12. Install the required libraries:

   `pip install transformers accelerate peft bitsandbytes datasets trl scikit-learn pandas`

13. Download the `crowdflower/twitter-airline-sentiment` dataset as the case study (registration required, free).

   `https://www.kaggle.com/datasets/crowdflower/twitter-airline-sentiment`  
   `https://www.kaggle.com/datasets/crowdflower/twitter-airline-sentiment`

   Click **Download** → **Download dataset as zip**. The CSV file is about 3 MB. Its contents look like this:

   ```text
   tweet_id    sentiment    author    content
   1956967341  empty        xoshayzers  @tiffanylue i know i was listenin to bad habit earlier and i started freakin at his part =[
   1956967666  sadness      wannamama   Layin n bed with a headache ughhhh...waitin on your call...
   ```

   This is a very well-known dataset, and it can also be replaced with any dataset you choose.

14. Unzip the archive directly and drag `twitter-airline-sentimentSentiment_Analysis.csv` into the same folder as the Jupyter notebook on the left side (`/root/`).


## 3. Download the base model

Download the model `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` locally to:

`/root/private_data/DeepSeek1.5B`


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Define local save directory for the 1.5B model
local_model_dir = "/root/private_data/DeepSeek1.5B"

# ✅ Correct Model Name (1.5 Billion parameters)
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

print(f"Loading model: {model_name} (Official size: 1.54B parameters)")

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,   # bfloat16 is safe and memory efficient
    device_map="auto"             # automatic device placement
)

# Save locally
tokenizer.save_pretrained(local_model_dir)
model.save_pretrained(local_model_dir)
print(f"Model saved to {local_model_dir}")


## 4. Official  notes

The following is an excerpt from the official  documentation:

`https://www..../help/docs/mainsite/ai/notebook/function-introduction/`

### 4.1 Save the environment when shutting down

You can save the development environment when shutting down, or use the **Save Image** feature to back it up. This helps ensure consistent environment configuration for restarts, team collaboration, and reproducing the environment on other platforms. You can save the image both when the container instance is on and when it is shut down.

### 4.2 Important storage note

To ensure the image works properly, a single-layer image should not exceed 15 GiB. The system validates the image size. If the image exceeds the limit, you must manually move container files to file storage.

### 4.3 Find large files in the current environment

```bash
cd /
find . -path "./proc" -prune -o        -path "/root/private_data/*" -prune -o        -path "/root/public_data/*" -prune -o        -path "/root/group_data/*" -prune -o        -path "/public/*" -prune -o        -path "/work/*" -prune -o        -type f -exec du -h {} + | sort -hr | head -n 20
```

This command shows the top 20 largest files and folders in the environment.

### 4.4 Move large files to persistent storage

```bash
mv /root/model_file /root/private_data/model_file
```

After migration, the file may not be usable in file storage because the owner is `root`. You need to update permissions in the current environment:

```bash
# Replace user_name with your actual username
chown user_name:user_name /root/private_data/model_file
```


## 5. Test the downloaded model


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

local_model_dir = "/root/private_data/DeepSeek1.5B"

tokenizer = AutoTokenizer.from_pretrained(local_model_dir, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    local_model_dir,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Example prompt
input_text = "Explain quantum computing in simple terms"
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


## 6. Load the dataset


In [ ]:
import pandas as pd

df = pd.read_csv('twitter-airline-sentimentSentiment_Analysis.csv')
first_50 = df.head(5000)
print(f"Loaded {len(first_50)} rows")
print(first_50[['tweet_id', 'sentiment', 'author', 'content']].head())


The CSV file must be placed in the same folder as the Jupyter notebook.


## 7. Prepare the fine-tuning data (use the first 5,000 rows)


In [ ]:
# Define instruction
instruction = "Analyze the sentiment of the following tweet:"

# Create a list of formatted texts
formatted_texts = []
for idx, row in first_50.iterrows():
    text = f"Instruction: {instruction}\nInput: {row['content']}\nOutput: {row['sentiment']}"
    formatted_texts.append(text)

# Convert to a Hugging Face Dataset
from datasets import Dataset
dataset = Dataset.from_dict({"text": formatted_texts})

print(dataset[0]['text'])


## 8. 4-bit quantization and model training setup


In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from datasets import Dataset

# 4‑bit quantization config (saves memory)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Load model with quantization
model_name = "/root/private_data/DeepSeek1.5B"  # or the HF name
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],   # typical for DeepSeek
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Wrap model with LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # should show ~0.1% trainable


## 9. Prepare the training data


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors=None  # we'll handle with data collator
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask"])

def set_labels(example):
    example["labels"] = example["input_ids"].clone()
    return example

tokenized_dataset = tokenized_dataset.map(set_labels)


## 10. Use the standard Hugging Face training workflow


In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # causal LM
)


## 11. Set the training arguments


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=3,
    # logging_steps=10,
    save_strategy="epoch",
    num_train_epochs=3,
    optim="paged_adamw_8bit",
    report_to="none"
)


## 12. Start training

This takes about 16 minutes.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()


## 13. Save the fine-tuned model


In [ ]:
output_dir = "/root/private_data/DeepSeek1.5B_finetuned"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"LoRA adapter saved to {output_dir}")


## 14. Run a demonstration with the fine-tuned model


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Paths
# base_model_name = "deepseek-ai/deepseek-llm-1.5b-base"   # or your local path if saved
base_model_name = "/root/private_data/DeepSeek1.5B"
adapter_path = "/root/private_data/DeepSeek1.5B_finetuned"

# Optional: 4‑bit quantization (same as during training)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token   # important for generation

# Load base model (with or without quantization)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,    # remove if you didn't use quantization
    device_map="auto",
    trust_remote_code=True
)

# Load LoRA adapter
model = PeftModel.from_pretrained(base_model, adapter_path)

# Switch to evaluation mode
model.eval()

# --- Test the model ---
# Example tweet input
tweet = "I love this new phone! It's amazing 😍"
instruction = "Analyze the sentiment of the following tweet:"

# Format the prompt exactly as during training
prompt = f"Instruction: {instruction}\nInput: {tweet}\nOutput:"

# Tokenize and move to same device as model
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate (limit new tokens to a short answer, e.g., sentiment label)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=20,            # sentiment label is short
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode only the newly generated part (skip the prompt)
generated_ids = outputs[0][inputs.input_ids.shape[1]:]   # take only new tokens
generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

print(f"Tweet: {tweet}")
print(f"Predicted sentiment: {generated_text}")


## 15. Compare the fine-tuned model with the original dataset


In [ ]:
import torch
import pandas as pd
import random
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# --------------------------
# 1. Paths and model loading
# --------------------------
# base_model_name = "deepseek-ai/deepseek-llm-1.5b-base"   # or your local path if saved
base_model_name = "/root/private_data/DeepSeek1.5B"
adapter_path = "/root/private_data/DeepSeek1.5B_finetuned"

# If you used 4‑bit quantization during training, load with same config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token   # important for generation

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,    # remove if you didn't use quantization
    device_map="auto",
    trust_remote_code=True
)

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()   # inference mode

# --------------------------
# 2. Load the original CSV
# --------------------------
csv_path = "twitter-airline-sentimentSentiment_Analysis.csv"   # adjust if needed
df = pd.read_csv(csv_path)

# Ensure we have the required columns
print(f"CSV loaded with {len(df)} rows. Columns: {df.columns.tolist()}")

# --------------------------
# 3. Randomly pick 5 tweets
# --------------------------
sample_rows = df.sample(n=5, random_state=42)   # fixed seed for reproducibility

instruction = "Analyze the sentiment of the following tweet:"

# --------------------------
# 4. Test each sample
# --------------------------
for idx, row in sample_rows.iterrows():
    tweet = row['content']
    actual_sentiment = row['sentiment']

    # Build prompt exactly as during training
    prompt = f"Instruction: {instruction}\nInput: {tweet}\nOutput:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the newly generated tokens (skip the prompt)
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    predicted_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    # Optional: clean up predicted text (remove trailing newline, etc.)
    predicted_text = predicted_text.split('\n')[0]   # take first line

    print("\n" + "="*60)
    print(f"Tweet: {tweet[:100]}...")
    print(f"Actual sentiment: {actual_sentiment}")
    print(f"Predicted sentiment: {predicted_text}")
    print("="*60)


## 16. Finish up

1. Go back to **Console** → **Notebook** → **Actions** and shut down the container to prevent extra charges.

2. In the left-side **Artificial Intelligence** → **File Management** area, you can download the fine-tuned `DeepSeek1.5B_finetuned` model.

At this point, the example of fine-tuning a billion-parameter model on a single card is complete.

These fine-tuned models can be deployed efficiently locally, greatly reducing both API latency and cost. The same method can be directly extended to tens-of-billions-scale models within 80 GB, such as 7B LLMs.

### Contact

For jobs or project collaboration: `yucongcai_business@outlook.com`  
For research-related inquiries: `yucongcai_research@outlook.com`
